# Solution — Task 01: Ticket Classification Prompt Design

## Setup

In [ ]:
from openai import OpenAI
import json, os

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "tickets.json")) as f:
    tickets = json.load(f)

def parse_json_safe(text: str) -> dict | None:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print(f"✓ Setup complete. {len(tickets)} tickets loaded.")

## Solution 1.1 — system_prompt

In [ ]:
system_prompt = """You are an expert customer support ticket classifier.

Your task: classify each support ticket by category and priority.

Categories:
- billing: payments, duplicate charges, refunds, invoices, subscription plans
- technical: bugs, errors, API failures, crashes, performance issues
- account: login, password resets, profile changes, permissions, security
- shipping: delivery, tracking, returns, wrong/missing items

Priorities:
- high: production down, data loss, financial loss, security breach
- medium: impaired functionality, non-critical bugs, workaround available
- low: questions, minor issues, feature requests, cosmetic problems

Respond ONLY with valid JSON. No other text.
{"category": "billing|technical|account|shipping", "priority": "high|medium|low"}
"""

assert len(system_prompt.strip()) >= 100
assert all(kw in system_prompt.lower() for kw in ["json", "category", "priority"])
assert all(c in system_prompt.lower() for c in ["billing", "technical", "account", "shipping"])
assert all(p in system_prompt.lower() for p in ["high", "medium", "low"])
print("✓ Task 1.1 passed")

## Solution 1.2 — user_template

In [ ]:
user_template = "Classify this support ticket:\n\n{ticket}\n\nReturn JSON only."

assert "{ticket}" in user_template
assert "test content" in user_template.format(ticket="test content")
print("✓ Task 1.2 passed")

## Solution 1.3 — classify_ticket()

In [ ]:
def classify_ticket(client, ticket_text: str) -> dict:
    """Classify a ticket using the LLM and return parsed JSON dict."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_template.format(ticket=ticket_text)},
        ],
        temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)

result = classify_ticket(client, tickets[0]["text"])
assert isinstance(result, dict) and "category" in result and "priority" in result
assert result["category"] in {"billing", "technical", "account", "shipping"}
assert result["priority"] in {"high", "medium", "low"}
print(f"✓ Task 1.3 passed")
print(f"  Got: {result}  |  Expected: {tickets[0]['category']}/{tickets[0]['priority']}")

## Solution 1.4 — Accuracy Test

In [ ]:
cat_correct = 0
pri_correct = 0
errors = []

for t in tickets:
    result = classify_ticket(client, t["text"])
    if result.get("category") == t["category"]:
        cat_correct += 1
    else:
        errors.append({"ticket": t["text"][:50], "expected": t["category"], "got": result.get("category")})
    if result.get("priority") == t["priority"]:
        pri_correct += 1

cat_acc = cat_correct / len(tickets)
pri_acc = pri_correct / len(tickets)

print(f"Category accuracy: {cat_correct}/{len(tickets)} = {cat_acc:.0%}")
print(f"Priority accuracy: {pri_correct}/{len(tickets)} = {pri_acc:.0%}")
if errors:
    print(f"Errors: {errors}")

assert cat_acc >= 0.70
print("✓ Task 1.4 passed")

## Solution 1.5 — CoT Variant

In [ ]:
cot_system_prompt = """You are a ticket classifier. Think step by step before classifying.

1. Identify the core problem
2. Choose the category (billing / technical / account / shipping)
3. Assess severity and set priority (high / medium / low)

Return JSON: {"reasoning": "<step-by-step>", "category": "...", "priority": "..."}
"""

def classify_with_cot(client, ticket_text: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": cot_system_prompt},
            {"role": "user",   "content": ticket_text},
        ],
        temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)

result = classify_with_cot(client, tickets[5]["text"])
assert "reasoning" in result and "category" in result and "priority" in result
assert len(result["reasoning"]) >= 20
assert result["category"] in {"billing", "technical", "account", "shipping"}
print(f"✓ Task 1.5 passed")
print(f"  Reasoning: {result['reasoning'][:120]}...")

## Done

In [ ]:
print("\n✓ All task_01 tests passed!")